# Crop Yield Capstone

**Group name:** Group 7  |  **Our crop:** Maize  |  **Date:** October 2026

---
## Part 1 — Load the data
**Weeks 1–2**

##### Step 1.1: Import Libraries

Let's bring in all the tools we need for the whole project. We will start with the basic data manipulation and visualization libraries and bring in machine learning tools later when we need them.

* **pandas** = for handling data
* **numpy** = for doing math
* **matplotlib/seaborn** = for making charts

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
print('Ready')

Ready


##### Step 1.2: Define Project Scope & Parameters

We chose **Maize** as our crop. Let's set that up.

We will also define a few important rules for our analysis upfront:
- **RANDOM_STATE**: keeps our random steps the same every time we run the code.
- **HISTORY_YEARS**: defines "normal" as the average of a country's previous 5 harvests.
- **BAD_YEAR_DROP**: defines a bad year as a harvest that drops more than 15% below normal.
- **TEST_SHARE**: reserves the most recent 20% of years to test our predictions.

In [15]:
# Tell the notebook what our crop is
CROP = 'Maize'

# Set our ground rules for the analysis
RANDOM_STATE = 42
HISTORY_YEARS = 5
BAD_YEAR_DROP = 0.15
TEST_SHARE = 0.20

print(f'Working on: {CROP}')

Working on: Maize


##### Step 1.3: Load the Dataset

We load `yield_df.csv` with pandas into a dataframe called `df`. The file sits in the same folder as this notebook, so we can read it directly by name. We then check how many rows and columns it has and preview the first five records.

In [16]:
raw = pd.read_csv('yield_df.csv')

print('Rows:', raw.shape[0])
print('Columns:', raw.shape[1])
raw.head()

Rows: 28242
Columns: 8


,Unnamed: 0,Area,Item,Year,hg/ha_yield,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
0,0,Albania,Maize,1990,36613,1485.0,121.0,16.37
1,1,Albania,Potatoes,1990,66667,1485.0,121.0,16.37
2,2,Albania,"Rice, paddy",1990,23333,1485.0,121.0,16.37
3,3,Albania,Sorghum,1990,12500,1485.0,121.0,16.37
4,4,Albania,Soybeans,1990,7000,1485.0,121.0,16.37


##### Step 1.4: Initial Data Inspection

Three questions we always ask:
1. What columns do we have?
2. What is missing?
3. What do the numbers look like?

In [17]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28242 entries, 0 to 28241
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Unnamed: 0                     28242 non-null  int64  
 1   Area                           28242 non-null  object 
 2   Item                           28242 non-null  object 
 3   Year                           28242 non-null  int64  
 4   hg/ha_yield                    28242 non-null  int64  
 5   average_rain_fall_mm_per_year  28242 non-null  float64
 6   pesticides_tonnes              28242 non-null  float64
 7   avg_temp                       28242 non-null  float64
dtypes: float64(3), int64(3), object(2)
memory usage: 1.7+ MB


In [18]:
raw.isna().sum()

,0
Unnamed: 0,0
Area,0
Item,0
Year,0
hg/ha_yield,0
average_rain_fall_mm_per_year,0
pesticides_tonnes,0
avg_temp,0


In [19]:
raw.describe().round(2)

,Unnamed: 0,Year,hg/ha_yield,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
count,28242.00,28242.00,28242.00,28242.00,28242.00,28242.00
mean,14120.50,2001.54,77053.33,1149.06,37076.91,20.54
std,8152.91,7.05,84956.61,709.81,59958.78,6.31
min,0.00,1990.00,50.00,51.00,0.04,1.30
25%,7060.25,1995.00,19919.25,593.00,1702.00,16.70
50%,14120.50,2001.00,38295.00,1083.00,17529.44,21.51
75%,21180.75,2008.00,104676.75,1668.00,48687.88,26.00
max,28241.00,2013.00,501412.00,3240.00,367778.00,30.65


#####Step 1.5 Check your crop is actually there
Before filtering, list all crops and check ours is there.

In [20]:
items = raw['Item'].value_counts()
print('Crops in the file (and how many rows each has):')
print(items.to_string())
print()
print('Is your crop there?', CROP in items.index)
assert CROP in items.index, f"'{CROP}' not found. Copy the exact text from the list above."

Crops in the file (and how many rows each has):
Item
Potatoes                4276
Maize                   4121
Wheat                   3857
Rice, paddy             3388
Soybeans                3223
Sorghum                 3039
Sweet potatoes          2890
Cassava                 2045
Yams                     847
Plantains and others     556

Is your crop there? True


#####Step 1.6 Filter to Maize
Keep only rows where Item == CROP





In [21]:
df = raw[raw['Item'] == CROP].copy()

print(f"{len(df):,} rows | {df['Area'].nunique()} countries | years {df['Year'].min()} to {df['Year'].max()}")
df.head()

4,121 rows | 92 countries | years 1990 to 2013


,Unnamed: 0,Area,Item,Year,hg/ha_yield,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
0,0,Albania,Maize,1990,36613,1485.0,121.0,16.37
6,6,Albania,Maize,1991,29068,1485.0,121.0,15.36
12,12,Albania,Maize,1992,24876,1485.0,121.0,16.06
18,18,Albania,Maize,1993,24185,1485.0,121.0,16.05
23,23,Albania,Maize,1994,25848,1485.0,201.0,16.96


Step 1.7 Facts For Three Observations

In [22]:
# How many rows per country-year?
rows_per_cy = df.groupby(['Area', 'Year']).size()

# How many different rainfall values per country?
rain_values_per_country = df.groupby('Area')['average_rain_fall_mm_per_year'].nunique()

# Convert yield from hectograms to tonnes per hectare
yield_t = df['hg/ha_yield'] / 10_000

OBS = {
    'rows': len(df),
    'country_years': len(rows_per_cy),
    'repeated_cy': int((rows_per_cy > 1).sum()),
    'max_repeats': int(rows_per_cy.max()),
    'countries': df['Area'].nunique(),
    'rain_constant': int((rain_values_per_country == 1).sum()),
    'y_min': yield_t.min(), 'y_med': yield_t.median(), 'y_max': yield_t.max(),
    'y_max_country': df.loc[yield_t.idxmax(), 'Area'],
    'y_min_country': df.loc[yield_t.idxmin(), 'Area'],
}
print(OBS)

{'rows': 4121, 'country_years': 2028, 'repeated_cy': 575, 'max_repeats': 22, 'countries': 92, 'rain_constant': 92, 'y_min': 0.0849, 'y_med': 2.5401, 'y_max': 20.7556, 'y_max_country': 'Qatar', 'y_min_country': 'Botswana'}


### Our Three Observations

**1. The file holds 4,121 maize rows but only 2,028 different country-years.**
575 country-years appear more than once (up to 22 times each), so duplicates must be handled before any model is trained.

**2. Rainfall is a single fixed number in all 92 countries.**
It never changes from year to year — so it identifies the country rather than describing a wet or dry season.

**3. Yields are very unequal: they range from 0.08 t/ha (Botswana) to 20.76 t/ha (Qatar), with a median of 2.54 t/ha.**
So a typical country and the best country are far apart.